In [22]:
import os
import json
import asyncio
from typing import List, Optional, Any
from datetime import datetime
from IPython.display import Markdown, display

from langgraph.types import Command
from langchain_core.runnables import RunnableConfig
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, ToolMessage
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END, MessagesState

In [2]:
from dotenv import load_dotenv
load_dotenv('.env')

True

In [3]:
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=4,  
    check_every_n_seconds=0.1,
    max_bucket_size=10,
)

basic_llm = init_chat_model("openai:gpt-5", rate_limiter=rate_limiter)
reasoning_llm = init_chat_model("openai:gpt-5", rate_limiter=rate_limiter)

In [4]:
#used in Plan which is used in State
class Step(BaseModel):
    title: str
    description: str = Field(..., description="Specify exactly what data to collect")
    execution_res: Optional[str] = Field(
        default=None, description="The Step execution result"
    )

#used in the State object
class Plan(BaseModel):
    thought: str
    title: str
    steps: List[Step] = Field(
        default_factory=list,
        description="Research & Processing steps to get more context",
    )


class State(MessagesState):
    """State for our research agents."""
    research_topic: str = ""
    observations: list[str] = []
    queries: list[str] = []
    plan_iterations: int = 0
    current_plan: Plan | str = None
    final_report: str = ""

In [5]:
plan_prompt = """Today's date is {{ date }}. You are a professional research agent. 
Use the tools at your disposal to research the user's question, do not rely on in-built knowledge or other context, be sure to thoroughly research a given topic and provide any caveats or other considerations with your results, including ways the results could be incomplete or misleading.

Information Standards and Success

A successful research plan must meet these requirements:

Comprehensive Coverage -Explore all aspects of the topic to identify associated themes or trends that shed light on the question at hand -Represent multiple viewpoints or perspectives

Detailed Analysis -For key portions of the question or topic, perform detailed analysis, performing research to further explore a specific area -Detailed data points, facts, and statistics are required -Multiple sources are required for in-depth analysis.

Tools available for use in answering the user's question are:
{{ tools }}

You can construct a plan that uses these tools iteratively, receiving results and planning subsequent steps or queries depending on the results.
"""


In [6]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools.structured import StructuredTool

client = MultiServerMCPClient(
    {
        "database": {
            "url": "http://localhost:8000/mcp",
            "transport": "streamable_http",
        }
    }
)
tools = await client.get_tools()

In [7]:
from jinja2 import Template

template = Template(plan_prompt)

rendered = template.render(date=datetime.now().strftime('%m/%d/%Y'), tools=tools)


In [8]:
print(rendered)

Today's date is 09/03/2025. You are a professional research agent. 
Use the tools at your disposal to research the user's question, do not rely on in-built knowledge or other context, be sure to thoroughly research a given topic and provide any caveats or other considerations with your results, including ways the results could be incomplete or misleading.

Information Standards and Success

A successful research plan must meet these requirements:

Comprehensive Coverage -Explore all aspects of the topic to identify associated themes or trends that shed light on the question at hand -Represent multiple viewpoints or perspectives

Detailed Analysis -For key portions of the question or topic, perform detailed analysis, performing research to further explore a specific area -Detailed data points, facts, and statistics are required -Multiple sources are required for in-depth analysis.

Tools available for use in answering the user's question are:
[StructuredTool(name='list_tables', descript

In [9]:
from dataclasses import dataclass, field, fields

#used in apply_prompt_templates
@dataclass(kw_only=True)
class Configuration:
    """The configurable fields."""
    max_plan_iterations: int = 1  # Maximum number of plan iterations
    max_step_num: int = 3  # Maximum number of steps in a plan
    mcp_settings: StructuredTool = None  # MCP settings, including dynamic loaded tools

    @classmethod
    def from_runnable_config(
        cls, config: Optional[RunnableConfig] = None
    ) -> "Configuration":
        """Create a Configuration instance from a RunnableConfig."""
        configurable = (
            config["configurable"] if config and "configurable" in config else {}
        )
        values: dict[str, Any] = {
            f.name: os.environ.get(f.name.upper(), configurable.get(f.name))
            for f in fields(cls)
            if f.init
        }
        return cls(**{k: v for k, v in values.items() if v})

In [10]:
question = "How did the MSRP of EVs in Washington State change over the years?"

initial_state = {
        #Setting up State
        "messages": [{"role": "user", "content": question}],
    }

config = {
    #runtime configuration or variables
        "configurable": {
            "thread_id": "default",
            "max_plan_iterations": 1,
            "max_step_num": 5,
            "mcp_settings": tools
        },
        "recursion_limit": 100,
    }

In [23]:
def prepare_prompt(node, config, messages, plan=None):
    with open(f"{node}.md", "r", encoding="utf-8") as file:
        plan_prompt = file.read()
    template = Template(plan_prompt)

    prompt = template.render(date=datetime.now().strftime('%d/%m/%Y'), tools=config.mcp_settings, messages = messages, plan=plan)
    return [{"role": "system", "content": prompt}] + messages

def planner_node(state: State, config: RunnableConfig):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    plan_llm = reasoning_llm.with_structured_output(Plan,method="json_mode")
    #if plan_iterations >= configurable.max_plan_iterations:
    #    return Command(goto="reporter")
    messages = prepare_prompt("planner",configurable,messages)
    full_response = ""
    response = plan_llm.invoke(messages)
    full_response = response.model_dump_json(indent=4, exclude_none=True)
    plan_response = json.loads(full_response)
    new_plan = Plan.model_validate(plan_response)
    return Command(
        update={
            "messages": [AIMessage(content=full_response, name="planner")],
            "current_plan": new_plan,
        },
        goto="research_coordinator",
    )

In [12]:
def research_coordinator_node(state: State):
    print("Nothing")
    """Research team node that collaborates on tasks."""
    pass

In [13]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", research_coordinator_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

In [14]:
events = list(agent.stream(input=initial_state, config=config, stream_mode="values"))

Nothing


In [15]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}

{'current_plan', 'messages'}

In [21]:
import pprint
pprint.pprint(snapshot.values['messages'][1].content)

('{\n'
 '    "thought": "We will interrogate the Washington EV registrations SQLite '
 'database to identify MSRP-related columns, then compute time-series trends '
 'of EV MSRP over time. We will examine both model-year and, if available, '
 'registration-year trends; compute overall and by EV type (BEV vs PHEV); '
 'assess data quality (missingness, duplicates via VIN), and optionally trim '
 'outliers. The plan includes adaptable SQL after learning actual table and '
 'column names.",\n'
 '    "title": "Plan to analyze how EV MSRP in Washington State changed over '
 'the years using the EV registrations database",\n'
 '    "steps": [\n'
 '        {\n'
 '            "title": "List all tables",\n'
 '            "description": "Use list_tables to discover table names and '
 'basic info. Collect the full list of tables to identify the primary EV '
 'registration table."\n'
 '        },\n'
 '        {\n'
 '            "title": "Inspect schema of candidate table(s)",\n'
 '            "des

In [31]:
async def researcher_node(state: State, config: RunnableConfig):
    configurable = Configuration.from_runnable_config(config)
    messages = state.get("messages")
    plan = state.get("current_plan")
    context = state.get("observations", [])
    plan_text = plan.model_dump_json(indent=2, exclude_none=True) if isinstance(plan, Plan) else str(plan)
    prompt = prepare_prompt("researcher", configurable, messages, plan=plan_text)
    llm = basic_llm.bind_tools(configurable.mcp_settings)
    ai_message = llm.invoke(prompt)
    new_messages = [ai_message]
    observations = []
    tool_map = {t.name: t for t in configurable.mcp_settings}
    for tc in ai_message.additional_kwargs.get("tool_calls", []):
        name = tc["function"]["name"]
        args = json.loads(tc["function"].get("arguments", "{}"))
        tool = tool_map.get(name)
        if tool:
            result = tool.arun(args)
            observations.append(ToolMessage(content=json.dumps(result, indent=2), name=name, tool_call_id=tc["id"]))
    return Command(update={"messages": new_messages, "observations":observations})

In [34]:
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

graph = StateGraph(State)

# Add nodes
graph.add_node("planner", planner_node)
graph.add_node("research_coordinator", researcher_node)
graph.set_entry_point("planner")
graph.add_edge("research_coordinator", END)  # Option in the future, to add another step and filter the documents retrieved using rerhank before writing the report

agent = graph.compile(checkpointer=memory)

events = []
async for s in agent.astream(input=initial_state, config=config, stream_mode="values"):
    message = s["messages"][-1]
    events.append(message)

TypeError: Object of type coroutine is not JSON serializable

In [ ]:
events

In [ ]:
snapshot = agent.get_state(config)
{k for k, v in snapshot.values.items()}